# 05. 현황 · 시험 도래 · 정합성

전부 **읽기 전용**입니다. Excel 을 띄우지 않으니 파일을 열어 둔 채로도 됩니다.

In [ ]:
# 이 셀을 먼저 실행하세요. 어디서 열어도 프로젝트를 찾습니다.
import sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent                      # notebooks/ 에서 열었을 때
assert (ROOT / "main.py").exists(), f"프로젝트를 찾지 못했습니다: {pathlib.Path.cwd()}"
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

def 표(머리, 행들):
    """리스트를 표로 보여 준다."""
    if not 행들:
        display(Markdown("_내용 없음_"))
        return
    md = "| " + " | ".join(str(h) for h in 머리) + " |\n"
    md += "|" + "|".join("---" for _ in 머리) + "|\n"
    for r in 행들:
        md += "| " + " | ".join("" if c is None else str(c) for c in r) + " |\n"
    display(Markdown(md))

from core.config import load_config

# 설정은 현재 폴더 -> 프로젝트 폴더 순으로 찾고, 없으면 예시에서 만들어 준다
후보 = [pathlib.Path.cwd() / "config.yaml", ROOT / "config.yaml"]
cfg = load_config(next((p for p in 후보 if p.exists()), None))
print("프로젝트:", ROOT)
print("설정 파일:", cfg.source)

## 1. 파일 현황

최신 차수 · 갑지 마지막 블록 열 · 수불부 누계를 출력합니다.

In [ ]:
from core.dump import dump_state

dump_state(cfg)

## 2. 시험 도래 판정

- **겉모양**: 업체별 최초 반입 1회, 이후 누적 200본 초과마다 1회
- **밀크**: 주 3회 목표 — 판정이 아니라 카운터만
- **BSCW/JSP**: 진행 중인 시험의 예정일 초과 여부

In [ ]:
from core.due_checker import DueChecker
from core.state import State

state = State(ROOT / "state.json")
alerts = DueChecker(cfg, state).check()

표(["업체", "상태", "마지막 시험", "이후 누적", "사유"],
   [(v.업체, {"due": "🔴 필요", "warn": "🟡 임박", "ok": "-"}[v.level],
     v.마지막시험, f"{v.누적}본", v.사유 or (f"{v.남은본}본 남음" if v.남은본 else ""))
    for v in alerts.all_vendors])

if alerts.milk:
    print(alerts.milk)
for p in alerts.pending:
    print(p)
for w in alerts.warnings:
    print(" !", w)

### 모르는 업체명 경고

`vendor_alias 에 추가하세요` 가 뜨면 **그 업체는 시험 대상에서 영원히 빠집니다.**
가장 위험한 실패 모드라 조용히 넘기지 않고 알립니다.
`config.yaml` 의 `vendor_alias` 에 표기 5종을 추가하세요.

## 3. 정합성 검사

시험번호 연속성 · 일지↔대장 일치 · 평균값 3자 일치 · 일련번호 연속 ·
BSCW 날짜 규칙 · 판정과 측정값의 모순을 봅니다.

In [ ]:
from core.consistency import ConsistencyChecker

findings = ConsistencyChecker(cfg, state).check_all()
오류 = [f for f in findings if f.수준 == "error"]
경고 = [f for f in findings if f.수준 == "warn"]

표(["수준", "파일", "위치", "내용", "제안"],
   [("🔴" if f.수준 == "error" else "🟡", f.파일, f.위치, f.내용,
     f.제안값 if f.고칠수있음 else "") for f in findings])
print(f"오류 {len(오류)}건 · 경고 {len(경고)}건")

## 4. 워크플로우 미완료

체크하지 않은 수동 단계가 남아 있는 회차입니다.

In [ ]:
미완 = state.open_workflows()
표(["회차", "미완료 항목"], [(k, ", ".join(v)) for k, v in 미완])
if not 미완:
    print("미완료 없음")